# Checked: Contracting

This page runs the checks that the Checked block of the
[Contracting](../docs/contracting.md) chapter claims: thirteen S-shapes over the
measles evaluation, three M-shapes over the model graph, and the counterexamples
that must fail. A reader need not take the chapter's word for it. Every line
of output below is produced by the code above it, the gate re-executes this
notebook and fails if the committed outputs differ, and each claim is an
`assert` that stops the notebook when it does not hold.

In [1]:
import sys; sys.path[:0] = ["notebooks", "."]  # the shared module lives beside this notebook
import checked

# The shapes the chapter's Checked block names, by file.
RECORD_SHAPES = ["S0-Record", "S0-Member", "S0-Parties", "S0-Roles", "S0-Independence", "S0-Access", "S0-Population", "S0-StatementOfWork", "S0-Mission", "S0-Need", "S0-Proposal", "S0-Layers", "S9-Acceptance"]
MODEL_SHAPES = ["M1-Parties", "M1-Obligation", "M5-Cardinality"]

## The record conforms

The record is `track/measles-evaluation.ttl` with the EPO vocabulary and the
model graph, loaded as the test suite loads it: one default graph, since the
shapes derive an item's step through the model (sheet 10-33). The shapes
graph holds exactly the thirteen named shapes
copied from `shapes/epo.shapes.ttl`; a name that is not a shape in that file
raises an error, so a misnamed claim cannot pass silently.

In [2]:
record = checked.record()
S = checked.shapes("shapes/epo.shapes.ttl", RECORD_SHAPES)
conforms, fired = checked.report("track/measles-evaluation.ttl", record, S)
assert conforms and not fired
checked.passed("the measles evaluation conforms to the thirteen S-shapes the chapter names")

track/measles-evaluation.ttl: conforms = True
  S0-Access            pass
  S0-Independence      pass
  S0-Layers            pass
  S0-Member            pass
  S0-Mission           pass
  S0-Need              pass
  S0-Parties           pass
  S0-Population        pass
  S0-Proposal          pass
  S0-Record            pass
  S0-Roles             pass
  S0-StatementOfWork   pass
  S9-Acceptance        pass
ok: the measles evaluation conforms to the thirteen S-shapes the chapter names


## A requirement set before the agreement

The chapter says a requirement set declared before the agreement was signed
fails the layer rule, `S0-Layers`. The counterexample is
`counterexamples/requirements-before-agreement.ttl`, run against the same
thirteen shapes. `S0-Parties` fires as well: it carries its own ordering
constraint, that the agreement precedes the requirement set (R-21), and the
layer rule (R-32) generalises it to every pinned item.

In [3]:
cx = checked.counterexample("requirements-before-agreement.ttl")
conforms, fired = checked.report("counterexamples/requirements-before-agreement.ttl", cx, S)
assert not conforms and "S0-Layers" in fired
checked.passed("requirements-before-agreement.ttl does not conform and fails S0-Layers")

counterexamples/requirements-before-agreement.ttl: conforms = False
  S0-Access            pass
  S0-Independence      pass
  S0-Layers            FAIL
  S0-Member            pass
  S0-Mission           pass
  S0-Need              pass
  S0-Parties           FAIL
  S0-Population        pass
  S0-Proposal          pass
  S0-Record            pass
  S0-Roles             pass
  S0-StatementOfWork   pass
  S9-Acceptance        pass
  S0-Layers at ev:service-agreement:
    S0 two layers (R-32): every item of the record pinned at the contract is generated no later than the requirement set, and every item pinned within the evaluation no earlier than the agreement.
  S0-Parties at ev:service-agreement:
    S0: the agreement precedes the requirement set: requirements are agreed under the contract, not before it (R-21).
ok: requirements-before-agreement.ttl does not conform and fails S0-Layers


## A population neither interviewed nor represented

`counterexamples/population-unrepresented.ttl` is the measles evaluation with
the county residents' representation removed: an affected population nobody
speaks for. It must fail `S0-Population` and nothing else among the thirteen.

In [4]:
cx = checked.counterexample("population-unrepresented.ttl")
conforms, fired = checked.report("counterexamples/population-unrepresented.ttl", cx, S)
assert not conforms and set(fired) == {"S0-Population"}
checked.passed("population-unrepresented.ttl does not conform and fails S0-Population only")

counterexamples/population-unrepresented.ttl: conforms = False
  S0-Access            pass
  S0-Independence      pass
  S0-Layers            pass
  S0-Member            pass
  S0-Mission           pass
  S0-Need              pass
  S0-Parties           pass
  S0-Population        FAIL
  S0-Proposal          pass
  S0-Record            pass
  S0-Roles             pass
  S0-StatementOfWork   pass
  S9-Acceptance        pass
  S0-Population at ev:county-residents:
    S0: every affected population is spoken for: a stakeholder representation attributed to the person who speaks for it (epo:representedBy); where the statement of work decided an interview, a stakeholder input attributed to the population exists, the representation used it, and the population names who engaged it (epo:responsibleParty) (R-21, R-40, R-49 series wiring; sheet 10-13).
ok: population-unrepresented.ttl does not conform and fails S0-Population only


A population the statement of work said would be interviewed, but the record only represents (ruling R-40): the decision was made, the evaluation did not realize it.

In [5]:
cx = checked.counterexample("engagement-mismatch.ttl")
conforms, fired = checked.report("counterexamples/engagement-mismatch.ttl", cx, S)
assert not conforms and set(fired) == {"S0-Population"}
checked.passed("engagement-mismatch.ttl does not conform and fails S0-Population only")

counterexamples/engagement-mismatch.ttl: conforms = False
  S0-Access            pass
  S0-Independence      pass
  S0-Layers            pass
  S0-Member            pass
  S0-Mission           pass
  S0-Need              pass
  S0-Parties           pass
  S0-Population        FAIL
  S0-Proposal          pass
  S0-Record            pass
  S0-Roles             pass
  S0-StatementOfWork   pass
  S9-Acceptance        pass
  S0-Population at ev:county-residents:
    S0: every affected population is spoken for: a stakeholder representation attributed to the person who speaks for it (epo:representedBy); where the statement of work decided an interview, a stakeholder input attributed to the population exists, the representation used it, and the population names who engaged it (epo:responsibleParty) (R-21, R-40, R-49 series wiring; sheet 10-13).
ok: engagement-mismatch.ttl does not conform and fails S0-Population only


## One person, two roles

Sheet 10-13: a person holds at most one of the person roles. In
`counterexamples/one-person-team.ttl` Annie holds the domain expert role and
the evaluation operator role at once. It must fail `S0-Roles` and nothing
else among the thirteen.

In [6]:
cx = checked.counterexample("one-person-team.ttl")
conforms, fired = checked.report("counterexamples/one-person-team.ttl", cx, S)
assert not conforms and set(fired) == {"S0-Roles"}
checked.passed("one-person-team.ttl fails S0-Roles and no other of the twelve")

counterexamples/one-person-team.ttl: conforms = False
  S0-Access            pass
  S0-Independence      pass
  S0-Layers            pass
  S0-Member            pass
  S0-Mission           pass
  S0-Need              pass
  S0-Parties           pass
  S0-Population        pass
  S0-Proposal          pass
  S0-Record            pass
  S0-Roles             FAIL
  S0-StatementOfWork   pass
  S9-Acceptance        pass
  S0-Roles at ev:annie:
    S0: a person holds at most one of the person roles: domain expert, evaluation operator, authorized representative, sponsor signatory (sheet 10-13; the disjointness axioms are on the role classes and do not reach persons).
ok: one-person-team.ttl fails S0-Roles and no other of the twelve


## An independence nobody declared

Sheet 10-07: the testing organization's independence of the test item
provider is declared, dated and attributed, not assumed.
`counterexamples/independence-undeclared.ttl` drops the declaration. It must
fail `S0-Parties` and nothing else among the thirteen.

In [7]:
cx = checked.counterexample("independence-undeclared.ttl")
conforms, fired = checked.report("counterexamples/independence-undeclared.ttl", cx, S)
assert not conforms and set(fired) == {"S0-Parties"}
checked.passed("independence-undeclared.ttl fails S0-Parties and no other of the twelve")

counterexamples/independence-undeclared.ttl: conforms = False
  S0-Access            pass
  S0-Independence      pass
  S0-Layers            pass
  S0-Member            pass
  S0-Mission           pass
  S0-Need              pass
  S0-Parties           FAIL
  S0-Population        pass
  S0-Proposal          pass
  S0-Record            pass
  S0-Roles             pass
  S0-StatementOfWork   pass
  S9-Acceptance        pass
  S0-Parties at ev:service-agreement:
    S0: the testing organization's independence is declared, not assumed (sheet 10-07): an independence declaration in the record, attributed to the authorized representative and dated, names the test item provider it is independent of; a testing organization that holds the test item provider's role declares no such independence.
ok: independence-undeclared.ttl fails S0-Parties and no other of the twelve


## An acceptance by the organization

Sheet 10-06: where the sponsor signs, a named person signs for it. In
`counterexamples/acceptance-by-organization.ttl` the acceptance is attributed
to the county public-health office rather than to its signatory. It must fail
`S9-Acceptance` and nothing else among the thirteen.

In [8]:
cx = checked.counterexample("acceptance-by-organization.ttl")
conforms, fired = checked.report("counterexamples/acceptance-by-organization.ttl", cx, S)
assert not conforms and set(fired) == {"S9-Acceptance"}
checked.passed("acceptance-by-organization.ttl fails S9-Acceptance and no other of the twelve")

counterexamples/acceptance-by-organization.ttl: conforms = False
  S0-Access            pass
  S0-Independence      pass
  S0-Layers            pass
  S0-Member            pass
  S0-Mission           pass
  S0-Need              pass
  S0-Parties           pass
  S0-Population        pass
  S0-Proposal          pass
  S0-Record            pass
  S0-Roles             pass
  S0-StatementOfWork   pass
  S9-Acceptance        FAIL
  S9-Acceptance at ev:acceptance-1:
    S9: the acceptance is the sponsor's signatory's act on behalf of the sponsor, follows the delivery it accepts, and is an item of the accept step (the step is derived through the model graph, sheet 10-33).
ok: acceptance-by-organization.ttl fails S9-Acceptance and no other of the twelve


## The model graph conforms

The model graph is `model/og-caie.model.ttl`, the pruned RDF rendering of the
SysML source (R-22). The three named M-shapes are copied from
`shapes/model.shapes.ttl`.

In [9]:
model = checked.model_graph()
M = checked.shapes("shapes/model.shapes.ttl", MODEL_SHAPES)
conforms, fired = checked.report("model/og-caie.model.ttl", model, M)
assert conforms and not fired
checked.passed("the model graph conforms to the three M-shapes the chapter names")

model/og-caie.model.ttl: conforms = True
  M1-Obligation        pass
  M1-Parties           pass
  M5-Cardinality       pass
ok: the model graph conforms to the three M-shapes the chapter names


## A model with no obligation

`counterexamples/model/no-obligation.sysml` names a sponsor and affected
populations but relates them by nothing, and its mission regards no
population. It is built to RDF through the same pipeline as the canonical
graph (the pinned OpenSysML converter, then `scripts/prune_model.py`), and
must fail `M1-Obligation`. The package is minimal, so its testing
organization declares no authorized representative and `M5-Cardinality` fires too;
the claim is about `M1-Obligation`, and the assert says exactly that.

In [10]:
cx = checked.model_counterexample("no-obligation.sysml")
conforms, fired = checked.report("counterexamples/model/no-obligation.sysml", cx, M)
assert not conforms and "M1-Obligation" in fired
checked.passed("no-obligation.sysml does not conform and fails M1-Obligation")

counterexamples/model/no-obligation.sysml: conforms = False
  M1-Obligation        FAIL
  M1-Parties           pass
  M5-Cardinality       FAIL
  M1-Obligation at elmt:NoObligation__Mission:
    The mission regards one or more affected populations: the Mission item owns a reference part typed AffectedPopulation with lower bound one (R-37, R-38).
  M1-Obligation at elmt:NoObligation__OgCaieEvaluation:
    The assembly relates the sponsor to the affected populations by an obligation: a connection usage typed Obligation whose ends are the sponsor part and the affected part, a relation that carries no item (R-38).
  M5-Cardinality at elmt:NoObligation__SponsorOrganization:
    The sponsor organization holds exactly one signatory (SponsorSignatory), the named person who signs, approves the requirement set and accepts on its behalf (sheet 10-06, R-50).
  M5-Cardinality at elmt:NoObligation__TestingOrganization:
    The testing organization holds exactly one authorized representative (Authori

## Verdict

One line for the reader and for the gate. It is printed only when every cell
above ran and every assert held.

In [11]:
checked.verdict()

claims checked: 9
  the measles evaluation conforms to the thirteen S-shapes the chapter names
  requirements-before-agreement.ttl does not conform and fails S0-Layers
  population-unrepresented.ttl does not conform and fails S0-Population only
  engagement-mismatch.ttl does not conform and fails S0-Population only
  one-person-team.ttl fails S0-Roles and no other of the twelve
  independence-undeclared.ttl fails S0-Parties and no other of the twelve
  acceptance-by-organization.ttl fails S9-Acceptance and no other of the twelve
  the model graph conforms to the three M-shapes the chapter names
  no-obligation.sysml does not conform and fails M1-Obligation
NOTEBOOK: PASS
